# Regime · 01 — Mars inference (combined geom + regime CNN)

**Primary interface** for the per-regime Mars inference step. This notebook
calls `channel_heads.*` only — the same functions the batch wrapper
`scripts/run_mars_combined_regime.py` uses:

- `channel_heads.inference.regime.attach_regime_embeddings` — overrides the
  baseline embeddings with the regime CNN's
- `channel_heads.inference` — `load_feature_columns`, `load_threshold`,
  `load_xgb_model`, `predict_with_threshold`, `pick_device`

It is **read-only**: it scores in memory and summarises. It writes no prediction
files — those are produced by the batch wrapper.

> The regime model artifacts live under `models/` (git-ignored). If the chosen
> regime's artifacts are absent, the scoring cell reports that and stops
> gracefully; the notebook is fully functional once they exist.

In [9]:
import pandas as pd

from channel_heads.config import PROJECT_ROOT
from channel_heads.inference import (
    load_feature_columns,
    load_threshold,
    load_xgb_model,
    pick_device,
    predict_with_threshold,
)
from channel_heads.inference.regime import attach_regime_embeddings
from channel_heads.regimes import REGIMES

REGIME_NAME = "regA"
regime = REGIMES[REGIME_NAME]

INPUT_PARQUET = PROJECT_ROOT / "data/Mars/model_inputs/mars_model_input_tabular_plus_cnn.parquet"
PATCH_INDEX_PARQUET = PROJECT_ROOT / "data/Mars/model_inputs/mars_cnn_patch_index.parquet"
CNN_PATH = PROJECT_ROOT / f"models/cnn_outlet_{regime.name}.pt"
XGB_PATH = PROJECT_ROOT / f"models/xgb_geom_plus_cnn_emb_{regime.name}.json"
FEAT_PATH = PROJECT_ROOT / f"models/feature_columns_geom_plus_cnn_emb_{regime.name}.txt"
THR_PATH = PROJECT_ROOT / f"models/optimal_threshold_geom_plus_cnn_emb_{regime.name}.txt"

artifacts = [CNN_PATH, XGB_PATH, FEAT_PATH, THR_PATH]
have_model = all(p.exists() for p in artifacts)
print(f"regime={regime.name}  inputs exist={INPUT_PARQUET.exists() and PATCH_INDEX_PARQUET.exists()}  "
      f"model artifacts present={have_model}")

regime=regA  inputs exist=True  model artifacts present=True


## Load the Mars tabular + CNN input (read-only)

In [10]:
df_in = pd.read_parquet(INPUT_PARQUET)
print(f"{len(df_in)} rows x {df_in.shape[1]} columns, "
      f"{df_in['network_id'].nunique()} networks")

3785 rows x 34 columns, 391 networks


## Attach regime embeddings + score — via the package

Overrides the baseline embeddings with the regime CNN, then runs the regime's
combined XGBoost. Skips gracefully if the regime artifacts are absent.

In [11]:
if not have_model:
    missing = [str(p.relative_to(PROJECT_ROOT)) for p in artifacts if not p.exists()]
    print("Regime model artifacts absent — skipping scoring. Missing:")
    for m in missing:
        print("  -", m)
    print(f"Run: python scripts/run_mars_combined_regime.py --regime {regime.name}")
else:
    device = pick_device()
    print("device:", device)
    df = attach_regime_embeddings(
        df_in, CNN_PATH, PATCH_INDEX_PARQUET, PROJECT_ROOT, device
    )
    feats = load_feature_columns(FEAT_PATH)
    threshold = load_threshold(THR_PATH)
    model = load_xgb_model(XGB_PATH)
    proba, pred = predict_with_threshold(model, df, feats, threshold)

    print(f"threshold={threshold:.6f}  features={len(feats)}")
    print(f"predicted touching: {int(pred.sum())}/{len(pred)} ({100*pred.mean():.1f}%)")
    print("prob summary:")
    print(pd.Series(proba).describe()[["min", "25%", "50%", "mean", "75%", "max"]].round(4))

device: mps
threshold=0.777315  features=9
predicted touching: 1826/3682 (49.6%)
prob summary:
min     0.0001
25%     0.4354
50%     0.7739
mean    0.6520
75%     0.9101
max     0.9986
dtype: float64


---
Full-dataset run (writes the per-regime prediction tables and GeoPackage):

```bash
python scripts/run_mars_combined_regime.py --regime regB
```